# Apache Impala

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/impala/impala.ipynb)

Apache Impala is a massively parallel (MPP) SQL query engine for data stored in HDFS, object storage, and Apache Kudu. It runs as long-lived daemons (`impalad`) on every DataNode, avoids MapReduce entirely, and returns interactive results over Hive-compatible tables.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

Impala itself needs a Hadoop cluster, so the runnable cells simulate its mechanisms in plain Python (join distribution, two-phase aggregation, the metadata cache) and use DuckDB to show Parquet row-group skipping.

## Architecture

**Mental model.** An Impala cluster is three long-lived services on top of Hadoop. `impalad` runs on every worker, accepts client SQL, plans it, and executes query fragments; whichever daemon takes your connection coordinates that query. `statestored` tracks the live `impalad` membership and pushes small state updates to them. `catalogd` owns the catalog: it loads table metadata from the Hive metastore, runs DDL, and broadcasts catalog changes to every `impalad`.

**How a query flows.** Client to coordinator `impalad` (port `21050` for HiveServer2/JDBC, `21000` for the beeswax shell protocol). The planner asks `catalogd` for table metadata and the NameNode for block locations, then dispatches plan fragments to `impalad` workers. Each fragment scans local HDFS blocks, results are exchanged between fragments, and the coordinator assembles and streams rows back. Per-daemon web UIs expose live state: `impalad` on `25000`, `statestored` on `25010`, `catalogd` on `25020`.

**Worked example and common mistake.**

```bash
# Point the shell at any impalad; that node coordinates this session
impala-shell -i coordinator-host:21000 -d analytics

# One-shot query with delimited output
impala-shell -i coordinator-host:21000 -B --output_delimiter=',' \
  -q "select count(*) from analytics.fact_orders"

# Run a script and keep a full log of statements and timings
impala-shell -i coordinator-host:21000 -f etl/refresh_orders.sql -o /tmp/refresh.log
```

Common mistake: treating `-i` as "the master node" to install or pin to. Every `impalad` can coordinate, so clients should connect through a load balancer or any healthy daemon, not one hard-coded host.

**Mental model.** The Hive metastore is the shared catalog service; Impala requires it. Hive itself (HiveServer2, the Tez or MapReduce execution engine) is optional and does not need to run on the same hosts. Many clusters run both against the same metastore: Spark or Hive writes, Impala reads.

**Gotcha.** People say "Hive is a requirement" and then debug the wrong process. The metastore is a standalone Thrift service (commonly port `9083`) with its own database; HiveServer2 is the query endpoint. Check the metastore when Impala cannot see tables, not HiveServer2.

### Failure handling

**Mental model.** There is no single point of failure for running queries. Any `impalad` can coordinate, and fragments run on many workers. If a worker dies, every query with a fragment there fails, and because queries are short you normally just rerun. Statestore loss does not stop queries but freezes membership updates. Catalogd loss lets DDL run locally but stops broadcast of catalog changes to other coordinators.

**Gotcha.** Monitor `catalog.ready` on the catalogd metrics endpoint. If it drops, other coordinators plan against stale metadata and users see "table not found" for tables that exist. Restart recovery repopulates state automatically, but the window matters during ETL.

## Query execution

### Why Impala is fast

**Mental model.** Four mechanisms stack. Runtime code generation uses LLVM to compile the query's inner loops to native code with constants folded in. Vectorized execution processes rows in batches, which amortizes branching and keeps data in cache. The columnar layout reads only the columns a query needs and skips row groups using min/max statistics. I/O scheduling spreads scan ranges across disks to keep them all busy.

**Gotcha.** Codegen has a fixed cost. On very short queries over small tables, compilation can dominate, so Impala has `disable_codegen_rows_threshold` (default 5000) to skip codegen below that row count. Setting `disable_codegen=true` globally is a diagnostic step for CPU or illegal-instruction errors, not a performance setting.

### Streaming versus blocking operators

**Mental model.** Operators are either streaming or blocking. `SCAN`, `FILTER`, and `HASH JOIN` probe sides stream rows as they arrive. `SORT`, `TOP-N` with a full ordering, `AGGREGATE` finalize, and `ANALYTIC` are blocking: they must consume all input before producing the first output row. The coordinator streams whatever the root operator emits.

**Gotcha.** `LIMIT` does not automatically make a blocking plan fast. If the plan contains a `SORT` or a final aggregation, the limit applies after that operator drains its input. Only a `TOP-N` operator or an early scan limit lets Impala stop reading sooner, so check the plan before promising instant first rows.

### Joins: broadcast versus partitioned

**Mental model.** Impala plans a join as a tree of `HASH JOIN` operators and picks a distribution method per join. A broadcast join ships the small build side to every probe node, so each node joins it against its local slice and no data is shuffled. A partitioned (shuffle) join hash-partitions both sides on the join key through an `EXCHANGE`, so matching rows land on the same node. Broadcast wins when the build side fits in memory; shuffle is the only option when both sides are large.

**How the choice is made.** The cost-based optimizer uses statistics to estimate the cardinality of each input after filters. It prefers broadcast when the build side is small relative to the probe side, and it also reorders joins so the smallest selective join runs first. `COMPUTE STATS` on both sides is what makes this decision good. Runtime filters are then built from the build side and pushed down to the probe scan, which cuts probe-side rows before the exchange.

```sql
-- Force broadcast only when the build side is genuinely small
select /* +BROADCAST */ o.order_id, r.region
from analytics.fact_orders o
join analytics.dim_region r on o.region_id = r.region_id;

-- Force a shuffle for two large tables
select /* +SHUFFLE */ a.order_id, b.reason
from analytics.fact_orders a
join analytics.fact_returns b on a.order_id = b.order_id;

-- Give the optimizer real numbers instead of hints
compute stats analytics.fact_orders;
compute stats analytics.dim_region;
```

Common mistake: leaving stats stale and then "fixing" the bad plan with a permanent broadcast hint. Hints override the planner, so a hint that was right at one million rows becomes a memory failure at one hundred million.

**Mental model.** For two large tables the planner inserts an `EXCHANGE` that repartitions both inputs on the join key. Each node then builds one partition of the build side in memory and streams its probe partition through it. Memory is bounded by the largest build partition, not the whole table, which is why shuffle joins scale to tables far larger than node memory.

**Gotcha.** Shuffle joins are sensitive to key skew. If one join key holds a large share of rows, one node builds a huge hash table while others idle, and that node spills or fails. Detect it in the profile by comparing `RowsProduced` and `PeakMemoryUsage` across fragment instances; fix it by filtering, salting the key, or pre-aggregating the skewed side.

**Runtime filters.** When Impala builds the hash table for a join, it also builds a filter on the join key (a bloom filter plus min/max bounds). In `global` mode that filter is routed to every probe-side scan fragment, so the probe side drops non-matching rows before they reach the join and before they cross an exchange. `SET RUNTIME_FILTER_MODE=global` is the default in modern Impala; `per_node` limits routing to nodes running both fragments, and `off` disables it.

The simulation below runs the same join both ways on four workers, counts rows that cross the network, and then adds a runtime filter.

In [ ]:
import zlib
from collections import Counter, defaultdict

NODES = 4
REGIONS = ["EU", "US", "APAC"]
dim_customer = [(cid, REGIONS[cid % 3]) for cid in range(300)]                  # small build side
fact_orders = [(oid, oid % 300, float(oid % 50)) for oid in range(12_000)]     # (order_id, customer_id, amount)


def node_of(key):
    return zlib.crc32(str(key).encode()) % NODES


def scatter(rows):
    """Where the scan finds rows: spread over workers by block placement, not by any key."""
    placed = defaultdict(list)
    for i, row in enumerate(rows):
        placed[i % NODES].append(row)
    return placed


fact_scan, dim_scan = scatter(fact_orders), scatter(dim_customer)
print("fact rows per node:", [len(fact_scan[n]) for n in range(NODES)])

In [ ]:
def broadcast_join(probe, build_rows):
    """Ship the whole build side to every node; the probe side never moves."""
    lookup = dict(build_rows)
    out = [(o, lookup[c], a) for n in range(NODES) for o, c, a in probe[n] if c in lookup]
    return out, len(build_rows) * NODES


def partitioned_join(probe, build_scan, key_filter=None):
    """Hash-partition both sides on the join key through an EXCHANGE, then join per node."""
    moved, probe_parts, build_parts = 0, defaultdict(list), defaultdict(dict)
    for n in range(NODES):
        for cid, region in build_scan[n]:
            target = node_of(cid)
            moved += target != n
            build_parts[target][cid] = region
        for o, c, a in probe[n]:
            if key_filter is not None and c not in key_filter:
                continue  # runtime filter applied at the scan, before the exchange
            target = node_of(c)
            moved += target != n
            probe_parts[target].append((o, c, a))
    out = [(o, build_parts[n][c], a) for n in range(NODES) for o, c, a in probe_parts[n] if c in build_parts[n]]
    return out, moved


broadcast_rows, broadcast_moved = broadcast_join(fact_scan, dim_customer)
partitioned_rows, partitioned_moved = partitioned_join(fact_scan, dim_scan)
assert sorted(broadcast_rows) == sorted(partitioned_rows)
assert broadcast_moved < partitioned_moved  # small dimension: broadcast moves far less

# WHERE c.region = 'EU': the build side is filtered, and its keys become a runtime filter for the probe scan.
eu_build = {n: [r for r in dim_scan[n] if r[1] == "EU"] for n in range(NODES)}
eu_keys = {cid for cid, region in dim_customer if region == "EU"}
no_filter_rows, no_filter_moved = partitioned_join(fact_scan, eu_build)
filtered_rows, filtered_moved = partitioned_join(fact_scan, eu_build, key_filter=eu_keys)
assert sorted(no_filter_rows) == sorted(filtered_rows)
assert filtered_moved < no_filter_moved / 2
print(f"rows moved: broadcast={broadcast_moved:,} partitioned={partitioned_moved:,}")
print(f"EU-only partitioned join: {no_filter_moved:,} rows moved without runtime filter, {filtered_moved:,} with")

### Aggregation

**Mental model.** Aggregation is hash-based and two-phase. A pre-aggregation runs where the scan happens to shrink the input, then an `EXCHANGE` repartitions by grouping key and a finalize `AGGREGATE` merges the partial states. This cuts shuffle volume dramatically. Memory is proportional to the number of distinct groups on a node, not to the number of input rows.

**Gotcha.** A `GROUP BY` on a near-unique column has almost no pre-aggregation benefit and a very large hash table. Use `ndv(col)` when an approximate distinct count is acceptable, and pre-aggregate high-cardinality groupings upstream in Spark or Hive. Watch `RowsSpilled` on the aggregate node as the signal that the group count exceeded memory.

In [ ]:
def two_phase_sum(scan, group_key):
    """Pre-aggregate where the scan runs, exchange partial states by group key, then finalize."""
    partials = {n: Counter() for n in range(NODES)}
    for n in range(NODES):
        for row in scan[n]:
            partials[n][group_key(row)] += row[2]
    exchanged = sum(len(p) for p in partials.values())  # partial rows crossing the network
    final = Counter()
    for partial in partials.values():
        final.update(partial)
    return final, exchanged


region_of = dict(dim_customer)
by_region, region_rows = two_phase_sum(fact_scan, lambda row: region_of[row[1]])
by_order, order_rows = two_phase_sum(fact_scan, lambda row: row[0])

single_node = Counter()
for _, cid, amount in fact_orders:
    single_node[region_of[cid]] += amount
assert by_region == single_node
assert region_rows == NODES * len(REGIONS)  # 12,000 input rows shrink to 12 partial rows
assert order_rows == len(fact_orders)       # a near-unique key gets no pre-aggregation benefit
print(f"partial rows exchanged: GROUP BY region={region_rows}, GROUP BY order_id={order_rows:,}")

### Scaling out

**Mental model.** Scan throughput scales close to linearly with workers because each one reads its local blocks. Everything else scales worse: a broadcast join copies the build side to every worker (fan-out grows with the cluster), a shuffle join pays network proportional to the data, and the coordinator alone does final aggregation, sorting, and result assembly.

**Gotcha.** Adding nodes does not fix a coordinator-bound query or a small-file table. If the profile shows most time in a single `EXCHANGE` or in planning over hundreds of thousands of files, more workers make it no faster. Watch `DataStreamSender` and `EXCHANGE` rows before assuming scale-out is the answer.

### Memory

**Mental model.** Per-query memory is not a function of table size. It comes from a few operators: scan buffers and decompression, hash tables for join build sides and `GROUP BY`, sort and analytic buffers, exchange buffers, and result batches. `impalad` allocates through `tcmalloc` and keeps freed memory cached for reuse. Two caps govern it: `--mem_limit` at daemon startup and the `MEM_LIMIT` query option; admission control estimates a query's memory before it is allowed to run.

**Worked example.** A 256 GB worker keeps roughly 80 percent for Impala, so `--mem_limit=200gb`. A dashboard query that peaks near 8 GB and a nightly aggregation that peaks near 40 GB mix as roughly `200 / 8` interactive slots or `200 / 40` heavy slots, but not both at once. Pools and `MEM_LIMIT` enforce that split. Common mistake: sizing memory from input bytes. A `GROUP BY` costs memory proportional to the number of groups, and a hash join costs memory proportional to the build side after filtering.

```sql
-- Per-query cap, useful when a pool is shared
set mem_limit = 16gb;

-- See what a query actually reserved and whether it spilled
set explain_level = 1;
```

Common mistake: leaving `MEM_LIMIT` unset in a shared pool. One bad query can then reserve the whole node and push every other query into the admission-control queue.

**Mental model.** Rank the cost drivers by operator state, not by input size: dynamic-partition `INSERT` into a Parquet table (one in-memory writer per open partition), `GROUP BY` on a high-cardinality key (one entry per distinct group), hash join build sides, `ORDER BY` without a limit (full sort state), and very wide rows with many `STRING` columns (up to 32 KB per value).

**Gotcha.** Dynamic partition inserts are the classic production incident. Loading one year at once can open hundreds of Parquet writers per node; loading one month at a time keeps memory flat. Cap the blast radius with `num_dynamic_partitions` and split the load by time range.

**Mental model.** Operators that support spilling write their in-memory state to a scratch directory on local disk and continue. In modern Impala this covers hash joins, hash aggregation, sorting, and analytic functions. Memory is bounded by `MEM_LIMIT` and by admission control; disk scratch is bounded by `SCRATCH_LIMIT` and the `--scratch_dirs` configuration.

**Gotcha.** Not every operator can spill. Nested-loop joins and some broadcast join builds cannot, so they fail with `Memory limit exceeded` instead of degrading. If the profile shows spill on every node, adding memory helps; if the failure is on a broadcast build side, changing the strategy to `SHUFFLE` is the real fix.

## Metadata and statistics

### The catalog, REFRESH and INVALIDATE METADATA

**Mental model.** Two metadata layers live inside `impalad`. The catalog layer (tables, columns, partitions, statistics, file format) comes from the Hive metastore through `catalogd`. The file layer (which files exist, their block locations and sizes) comes from the HDFS NameNode. Both are loaded lazily on first use and cached, so planning does not hit the metastore on every query.

**Refresh versus invalidate.** `REFRESH` reloads the file/block metadata for known tables, which is what you need after an external writer adds files. `INVALIDATE METADATA` discards and reloads the catalog structure, which is what you need after external DDL such as `CREATE TABLE`, `ALTER TABLE`, or `ADD PARTITION`. Both are broadcast by `catalogd` to every `impalad`, so one node can issue them.

**Decision rule.** New files in an existing table or partition: `REFRESH`. New table, renamed column, added or dropped partition, changed format: `INVALIDATE METADATA`. Both are broadcast to all nodes, so issue on one coordinator. Scope to one table whenever possible.

```sql
-- External writer added files to an existing partition
refresh analytics.fact_orders partition (order_date = '2026-10-04');

-- Hive created a brand new table (or changed its columns)
invalidate metadata analytics.new_table;

-- Make DDL wait until all coordinators have the change
set sync_ddl = true;
```

Common mistake: running `INVALIDATE METADATA` cluster-wide instead of `REFRESH` on one table. The full invalidate re-reads everything from the metastore and NameNode and can stall planning for minutes.

**Why DDL is slow.** Every catalog change is serialized through `catalogd`, written to the metastore, and broadcast to all `impalad` nodes; with `SYNC_DDL=true` the statement also waits for every node to acknowledge. `INVALIDATE METADATA` on a large table re-fetches every partition and file from the metastore and NameNode, which is why it should be scoped to one table and run off-peak. Common mistake: running `INVALIDATE METADATA` with no table name during business hours and stalling the whole cluster's planning.

In [ ]:
import tempfile
from pathlib import Path

WAREHOUSE = Path(tempfile.mkdtemp(prefix="impala_warehouse_"))


class Catalog:
    """A coordinator's metadata cache: table list and file listings, loaded lazily and reused."""

    def __init__(self, warehouse):
        self.warehouse = warehouse
        self.tables = {p.name: None for p in warehouse.iterdir()}  # table names known at startup
        self.file_cache = {}

    def files(self, table):
        if table not in self.tables:
            raise LookupError(f"Table does not exist: {table}")
        if table not in self.file_cache:  # first use loads file metadata from the NameNode
            self.file_cache[table] = sorted(f.name for f in (self.warehouse / table).rglob("*.parq"))
        return self.file_cache[table]

    def refresh(self, table):
        """REFRESH: reload file metadata for a table the catalog already knows."""
        if table not in self.tables:
            raise LookupError(f"Table does not exist: {table}")
        self.file_cache.pop(table, None)

    def invalidate_metadata(self, table):
        """INVALIDATE METADATA: forget everything about the table and rediscover it."""
        self.file_cache.pop(table, None)
        if (self.warehouse / table).exists():
            self.tables[table] = None


def external_write(table, partition, name):
    """A Spark or Hive job writing files behind Impala's back."""
    folder = WAREHOUSE / table / partition
    folder.mkdir(parents=True, exist_ok=True)
    (folder / name).write_text("parquet bytes")


external_write("fact_orders", "order_date=2026-10-03", "part-0.parq")
catalog = Catalog(WAREHOUSE)
assert catalog.files("fact_orders") == ["part-0.parq"]

external_write("fact_orders", "order_date=2026-10-04", "part-1.parq")
assert catalog.files("fact_orders") == ["part-0.parq"]  # stale: new partition invisible
catalog.refresh("fact_orders")
assert catalog.files("fact_orders") == ["part-0.parq", "part-1.parq"]

external_write("dim_customer", "all", "part-0.parq")  # new table created outside Impala
try:
    catalog.refresh("dim_customer")
    raise AssertionError("REFRESH cannot discover a new table")
except LookupError as exc:
    print("REFRESH on a new table:", exc)
catalog.invalidate_metadata("dim_customer")
assert catalog.files("dim_customer") == ["part-0.parq"]
print("new files need REFRESH; new tables need INVALIDATE METADATA")

### COMPUTE STATS

**Mental model.** The optimizer is cost-based and needs cardinality estimates. `COMPUTE STATS` scans the table once and writes row counts, file counts, sizes, and per-column statistics (number of distinct values, nulls, average and max size) into the metastore. The planner then estimates filter selectivity and join sizes from those numbers.

**Full versus incremental.** `COMPUTE STATS` rescans the whole table. `COMPUTE INCREMENTAL STATS` maintains per-partition statistics and only rescanning changed partitions, so on a large partitioned table it turns a multi-hour job into a per-partition one. Incremental stats must be enabled by running the full command once; after that, `DROP STATS` removes them. If a partition has no incremental stat, the planner falls back to the table-level estimate.

```sql
-- One-time full scan, then per-partition maintenance
compute stats analytics.fact_orders;
compute incremental stats analytics.fact_orders partition (order_date = '2026-10-04');

show table stats analytics.fact_orders;
show column stats analytics.fact_orders;

-- Remove stale stats after a schema change
drop stats analytics.fact_orders;
```

Common mistake: collecting stats only on the fact table. The dimension side also feeds the estimate, so a missing `COMPUTE STATS` on the small table can leave the planner guessing and broadcasting the wrong side.

**Common mistake.** Loading a new partition and skipping statistics. The planner keeps using the old row count, decides the table is smaller than it is, and broadcasts it. A tell-tale sign is a `BROADCAST` join on a table that has grown for months while `SHOW TABLE STATS` still reports the row count from the last full collection. Refresh the metadata after external writes, then recompute the affected statistics.

```sql
-- Full collection once, then maintain per partition
compute stats analytics.fact_orders;
compute incremental stats analytics.fact_orders partition (order_date = '2026-10-04');

-- Inspect what the planner believes
show table stats analytics.fact_orders;
show column stats analytics.fact_orders;
```

## Tables, formats and partitions

### DDL and file formats

**Mental model.** File format determines scan cost and capabilities. Parquet is columnar with per-row-group min/max statistics and dictionary encoding, which enables projection and predicate pushdown; ORC is similar and common in Hive-centric clusters. Avro and text are row-oriented landing formats. Kudu is a mutable, primary-key table format rather than a file format. Iceberg adds table-level snapshots on top of Parquet or ORC files.

**How pushdown works.** Impala reads only the referenced columns, then skips row groups whose statistics prove that no row can match the predicate. A function wrapped around a column hides the comparison, so `cast(date_col as string) = '2026-10-04'` defeats pushdown where `date_col = '2026-10-04'` uses it.

```sql
create table analytics.fact_orders (
  order_id bigint, customer_id bigint, status string,
  amount decimal(12,2), order_ts timestamp
)
partitioned by (order_date date)
stored as parquet
tblproperties ('parquet.compression' = 'zstd');

-- Future writes use the new codec; existing files keep their own
alter table analytics.fact_orders set tblproperties ('parquet.compression' = 'zstd');

-- Check what a table actually is
describe formatted analytics.fact_orders;
```

Common mistake: choosing a format for storage reasons only. Text compresses poorly and cannot be pushed down, so it should never be the format analysts query repeatedly.

Row-group statistics only help when the data is clustered on the filtered column:

In [ ]:
import duckdb

con = duckdb.connect()
parquet_path = WAREHOUSE / "orders_sorted.parquet"
ROWS_PER_DAY = 2048
con.execute(f"""
    COPY (
        SELECT i AS order_id,
               DATE '2026-10-01' + CAST(i // {ROWS_PER_DAY} AS INTEGER) AS order_date,
               (i * 7919 % 500)::DOUBLE AS amount
        FROM range(10 * {ROWS_PER_DAY}) t(i)
        ORDER BY order_id
    ) TO '{parquet_path}' (FORMAT parquet, ROW_GROUP_SIZE {ROWS_PER_DAY})
""")


def row_group_stats(column):
    return con.execute(f"""
        SELECT stats_min, stats_max FROM parquet_metadata('{parquet_path}')
        WHERE path_in_schema = '{column}' ORDER BY row_group_id
    """).fetchall()


def skippable(column, value, cast):
    """Row groups whose min/max prove that `column = value` cannot match."""
    return sum(not (cast(lo) <= value <= cast(hi)) for lo, hi in row_group_stats(column))


groups = len(row_group_stats("order_date"))
date_skips = skippable("order_date", "2026-10-04", str)
amount_skips = skippable("amount", 123.0, float)
matches = con.execute(f"SELECT count(*) FROM '{parquet_path}' WHERE order_date = DATE '2026-10-04'").fetchone()[0]

assert matches == ROWS_PER_DAY
assert date_skips >= groups - 2  # sorted column: tight ranges, nearly every row group skipped
assert amount_skips == 0          # scattered column: every range covers the value, nothing skipped
print(f"{groups} row groups; order_date filter skips {date_skips}, amount filter skips {amount_skips}")

### HDFS block size versus Parquet file size

**Mental model.** Three sizes are easy to confuse. The HDFS block size (commonly 128 MB) is how the file is spread across DataNodes. The Parquet file size is set when Impala writes, via `PARQUET_FILE_SIZE` (default 256 MB in Impala 2.0 and later). The Parquet row group inside that file is the unit whose statistics enable predicate pushdown. Impala never rewrites HDFS block settings.

**Gotcha.** Making HDFS blocks tiny does not speed up scans; it raises NameNode metadata and open costs. Control the Parquet file size instead.

```sql
set parquet_file_size = 256m;   -- Impala writes multi-block Parquet files
```

The underlying HDFS block size stays as the cluster configured it, and one Parquet file simply spans several HDFS blocks.

### Partitioned tables

**Mental model.** A partitioned table stores one directory per partition value, and the partition columns are encoded in the directory path rather than in the files. The planner prunes directories using the predicates in the query, so a filter on a partition column removes entire directories from the scan before any file is opened.

**How partitions get created.** Static partitions are named explicitly with `ALTER TABLE ... ADD PARTITION` or a literal `PARTITION` clause in an insert. Dynamic partitions come from the last projected columns of an insert and are created on the fly; `num_dynamic_partitions` caps how many one statement may create. Use dynamic for bulk loads and static for controlled backfills.

```sql
-- Add one partition explicitly
alter table analytics.fact_events
  add partition (event_month = '2026-10', event_day = '2026-10-04');

-- Dynamic insert: one directory per month/day in the input
set num_dynamic_partitions = 400;
insert into analytics.fact_events partition (event_month, event_day)
select event_id, user_id, event_ts, payload,
       date_format(event_ts, 'yyyy-MM'), date_format(event_ts, 'yyyy-MM-dd')
from staging.events_raw;

-- Dropping a partition removes its directory for managed tables
alter table analytics.fact_events drop partition (event_month = '2025-01');
```

Common mistake: partitioning by a high-cardinality column such as `user_id` or a raw timestamp. Thousands of partitions are manageable; millions produce metadata bloat, slow planning, and thousands of tiny files.

### Loading CSV and INSERT ... SELECT

**Mental model.** Text CSV is a landing format, not a query format. Stage it as an external text table over the incoming directory, then `INSERT ... SELECT` into a typed partitioned Parquet table. The `PARTITION` clause maps the last projected columns to partition keys. Dynamic partitioning creates one directory per distinct value in one pass.

**Gotcha.** CSV files often carry a header row and untyped nulls, and a single `INSERT` over many partition values can exhaust memory. Handle the header with a table property, cast explicitly, and load in bounded date ranges.

```sql
create external table staging.orders_csv (
  order_id string, customer_id string, amount string, order_date string
)
row format delimited fields terminated by ','
tblproperties ('skip.header.line.count' = '1')
location '/data/incoming/orders/';

set num_dynamic_partitions = 400;
insert into analytics.fact_orders partition (order_date)
select cast(order_id as bigint), cast(customer_id as bigint),
       cast(amount as decimal(12,2)), cast(order_date as date)
from staging.orders_csv
where order_date between '2026-01-01' and '2026-01-31';

compute stats analytics.fact_orders;
```

Common mistake: loading everything in one statement, then discovering that the header row became a row with null keys and that the insert exceeded memory.

**Mental model.** In a positional `INSERT ... SELECT`, the projected columns must line up with the target table's columns, and the partition key columns come last because they are not stored inside the data files. An explicit column list in the `INSERT` removes the ordering requirement entirely.

**Worked example and common mistake.**

```sql
-- Explicit and order-independent
insert into analytics.fact_orders (order_id, customer_id, amount, order_date)
select order_id, customer_id, amount, order_date
from staging.orders_clean;
```

Common mistake: `INSERT ... SELECT *` where the source puts the partition key in the middle. The insert either fails with a column-count mismatch or writes the wrong value into the wrong column. Use the explicit list or a view that reorders columns.

### Why INSERT statements fail

**Mental model.** An `INSERT` into a partitioned table opens a writer per output partition per node. With dynamic partitioning and many partitions, the coordinator and the DataNodes run out of file handles, threads, or memory before HDFS runs out of space. Parquet worst case: each partition's data is buffered in memory until it reaches `PARQUET_FILE_SIZE`.

**Worked example and common mistake.**

```sql
set num_dynamic_partitions = 400;   -- cap partitions touched by one INSERT
set num_scanner_threads = 8;        -- cap concurrent scan threads per node

insert into analytics.fact_orders partition (order_date)
select order_id, customer_id, amount, cast(order_date as date)
from staging.orders_clean
where order_date between '2026-01-01' and '2026-01-31';
```

Common mistake: one `INSERT` that creates five years of daily partitions. Split it by month or year so each statement opens a bounded number of writers. Also avoid `CASE` expressions over the projected columns, which complicate type matching and distribute badly.

### Row-level change: Kudu, Iceberg and partition overwrite

**Mental model.** The answer depends on the storage format. Plain HDFS/Parquet tables are immutable files, so change is expressed as `INSERT OVERWRITE` of a partition or table. Kudu tables have primary keys and support real `UPDATE`, `DELETE`, and `UPSERT`. Iceberg tables support row-level `UPDATE` and `DELETE` in Impala 4.3 and later.

**Worked example and common mistake.**

```sql
-- Idempotent bulk change on immutable files: overwrite one partition
insert overwrite analytics.fact_orders partition (order_date = '2026-10-04')
select order_id, customer_id, upper(status), amount
from staging.orders_fixed
where order_date = '2026-10-04';

-- Row-level change on Kudu, targetable by primary key
update analytics.customer_kudu set tier = 'platinum' where customer_id = 1001;
```

Common mistake: using `INSERT INTO` for a correction. It appends a second copy of the rows. Use `INSERT OVERWRITE` scoped to the partition so reruns are safe.

**Mental model.** Parquet tables are directories of immutable files optimized for large scans. Kudu tables are live distributed tables split into tablets by a partition schema, replicated for durability, and addressable by primary key. Kudu keeps rows in a columnar on-disk format with an in-memory delta store, so both key lookups and range scans are fast.

**How the partition schema works.** Kudu distributes rows by hash partitions on key columns to spread writes, and by range partitions on a key column to bound scans and allow cheap range drops. A well-designed schema hashes on the first key column and ranges on a time column. Kudu rows are updated in place, so `UPDATE`, `DELETE`, and `UPSERT` are cheap when the primary key is fully specified.

```sql
create table analytics.customer_kudu (
  customer_id bigint, name string, tier string, updated_at timestamp,
  primary key (customer_id)
)
partition by hash (customer_id) partitions 16
stored as kudu;

upsert into analytics.customer_kudu (customer_id, name, tier, updated_at)
values (1001, 'Ada Lovelace', 'gold', now());

update analytics.customer_kudu set tier = 'platinum' where customer_id = 1001;
```

Common mistake: putting multi-billion-row fact data in Kudu because upserts are convenient. Kudu capacity is bounded by local disks on its tablet servers and it is not the cheapest place for large analytical scans; keep facts in Parquet and dimensions in Kudu.

**Gotcha.** Kudu `UPDATE` and `DELETE` need equality predicates on every primary key column so the statement can target exact rows. A predicate on a non-key column forces a scan and can be rejected or extremely slow. On plain Parquet, the idempotent pattern is `INSERT OVERWRITE` of one partition, not a row update.

### Small files and maintenance

**Mental model.** Every file adds planning work, a NameNode lookup, and an open/seek/read cost. A partition with ten 200 MB files is efficient; a partition with ten thousand 200 KB files spends more time in metadata than in scanning. Small files usually come from streaming micro-batches or many concurrent inserts into dynamic partitions.

**How to fix and prevent it.** Rewrite the partition through `INSERT OVERWRITE` with a larger `PARQUET_FILE_SIZE`, which compacts in one pass, then `REFRESH` and collect stats. Prevent recurrence by writing 128-512 MB files from the upstream job, preferring partition overwrite to appends, and running periodic compaction on the busiest partitions.

```sql
set parquet_file_size = 512m;

insert overwrite analytics.fact_orders partition (order_date = '2026-10-04')
select order_id, customer_id, status, amount, order_ts
from analytics.fact_orders where order_date = '2026-10-04';

refresh analytics.fact_orders partition (order_date = '2026-10-04');
compute incremental stats analytics.fact_orders partition (order_date = '2026-10-04');
```

Common mistake: compacting without updating statistics. The file count changes, and the planner keeps using the old numbers until stats are recomputed.

### DROP TABLE and disk space

**Mental model.** `DROP TABLE` deletes data files only for managed (internal) tables. External tables own only the metadata, so their files stay where they are. `DESCRIBE FORMATTED` shows `MANAGED_TABLE` or `EXTERNAL_TABLE` in the `Table Type` row. Even for managed tables, files usually go to the HDFS trash and are purged after `fs.trash.interval`.

**Worked example and common mistake.**

```sql
-- Check before dropping, then decide how to clean up
describe formatted analytics.fact_orders;

-- Managed table: skip the trashcan and reclaim space immediately
drop table if exists analytics.fact_orders purge;

-- External table: drop metadata, then remove the path yourself
drop table if exists analytics.orders_ext;
-- hdfs dfs -rm -r -skipTrash /warehouse/analytics/orders_ext
```

Common mistake: dropping the external table and assuming the space is gone. Verify the location with `DESCRIBE FORMATTED`, then remove the directory explicitly once nothing else uses it.

### Caching

**Mental model.** There are three separate caches. The metadata cache holds table and file metadata inside `impalad` so planning does not hit the metastore every time. The OS page cache holds recently read blocks and makes repeat scans faster. The HDFS caching layer pins chosen data in memory on each DataNode, requested through `CACHED` and `UNCACHED` clauses on `CREATE TABLE` and `ALTER TABLE`.

**Gotcha.** Impala has no result cache. Re-running the same query returns the same rows by recomputing them, only faster if the OS or HDFS cache is warm. On object storage there is no OS pinning, so use the storage layer's cache or materialize hot aggregates into Kudu.

### UDFs

**Mental model.** Scalar UDFs and UDAs are native shared libraries loaded into every `impalad`; a Java Hive UDF can be reused for scalar work through its class name. UDFs run in the query hot path, inside the codegen'd pipeline, so they are compiled and called for every row. There is no UDTF support and no SerDes extension point.

**Gotcha.** A UDF that throws or returns nulls inconsistently can fail queries on one node and succeed elsewhere, which makes debugging hard. Keep UDFs small, deterministic, and side-effect free, and test them against the built-in equivalent before shipping.

## Operations

### Query optimization checklist

**Mental model.** Optimize in this order: format and layout, then statistics, then the plan, then memory. Format and layout decide how much data is read. Statistics decide whether the plan is good. The plan decides how much data moves. Memory and admission control decide whether the query runs at all under concurrency.

**Worked example.** A dashboard query was scanning three years of orders because the filter used `cast(order_date as string) = '2026-10-04'` instead of a date predicate. Rewriting it as `order_date = '2026-10-04'` restored partition pruning, and `COMPUTE STATS` let the optimizer pick a broadcast join for the small dimension.

```sql
compute stats analytics.fact_orders;
compute incremental stats analytics.fact_orders partition (order_date = '2026-10-04');

explain
select c.region, count(*) as orders, sum(o.amount) as revenue
from analytics.fact_orders o
join analytics.dim_customer c on o.customer_id = c.customer_id
where o.order_date = '2026-10-04'
group by c.region;
```

Common mistake: tuning the query before fixing the layout. No hint makes a million tiny files fast, and no plan helps when the filter prevents partition pruning.

### Reading EXPLAIN and PROFILE

**Mental model.** `EXPLAIN` prints the physical plan tree with estimated cardinality and, for joins, the chosen distribution. `PROFILE` (or the query profile UI) prints what actually happened, operator by operator, including rows produced, peak memory, and spill. Read the estimate first, then the reality, and compare.

**How to read it.** Nodes are numbered and indented under `PLAN-ROOT SINK`. `SCAN HDFS` lines show `partitions=`, `files=`, and `size=`. `HASH JOIN` lines show `[BROADCAST]` or `[PARTITIONED]` and the join predicates. `EXCHANGE` shows how rows move between fragments. In the profile, compare estimated versus actual rows per operator, then look at `TotalTime`, `PeakMemoryUsage`, `RowsSpilled`, and `BytesSpilled`.

**How to read the plan.** Start at the `SCAN HDFS` lines and confirm pruning with `partitions=` and `files=`. Move upward and check each `HASH JOIN` for `[BROADCAST]` versus `[PARTITIONED]`. `EXCHANGE` nodes are where data moves between fragments, so their row counts are network cost. In the profile, compare estimated with actual rows per operator and look at `TotalTime`, `PeakMemoryUsage`, `RowsSpilled`, and `BytesSpilled`.

```text
# Abbreviated shape of a partitioned join plan
03:EXCHANGE [UNPARTITIONED]
02:HASH JOIN [INNER JOIN, PARTITIONED]
|  hash predicates: o.customer_id = c.customer_id
|  runtime filters: RF000 <- c.customer_id
|--01:SCAN HDFS [analytics.dim_customer c]  partitions=1/1 files=4 size=12MB
00:SCAN HDFS [analytics.fact_orders o]      partitions=1/365 files=96 size=1.9GB
    predicates: o.order_date = '2026-10-04'
```

```sql
set explain_level = 3;   -- more detail on expressions and predicates

explain
select c.region, count(*) as orders
from analytics.fact_orders o
join analytics.dim_customer c on o.customer_id = c.customer_id
where o.order_date = '2026-10-04'
group by c.region;
```

Common mistake: reading `EXPLAIN` top-down and stopping at the first join. Start at the scan lines to confirm pruning, then follow the rows upward to find the operator whose output is far smaller than its input, which is where the real work happens.

**Common mistake.** Blaming the scan because it is the largest node in the tree. Scans usually stream steadily and cheaply; the expensive operator is the one where output cardinality collapses or explodes. Find the operator whose `actual_rows` diverges most from its estimate, then fix the statistics, the predicate, or the join strategy behind it.

### Resource pools and admission control

**Mental model.** Admission control is a gate in front of query execution. The planner estimates the memory a query needs, then the coordinator checks the target pool's limits against what is already running. The query starts immediately, waits in the pool queue, or is rejected. Pools are defined in the fair-scheduler XML and referenced per session or per query with `REQUEST_POOL`.

**How it is enforced.** Each pool has `maxResources` (memory and CPU) and `maxRunningApps` (concurrent queries). A query also has its own `MEM_LIMIT`. Waiting is bounded by `RESERVATION_REQUEST_TIMEOUT`; if no slot frees up in time, the query fails rather than waiting forever. `impalad` re-estimates at runtime and can cancel a query that exceeds its limit.

```text
<queue name="root">
  <queue name="bi">
    <maxResources>40000 mb</maxResources>
    <maxRunningApps>20</maxRunningApps>
  </queue>
  <queue name="etl">
    <maxResources>80000 mb</maxResources>
    <maxRunningApps>5</maxRunningApps>
  </queue>
</queue>
```

```sql
set request_pool = 'bi';
set mem_limit = 16gb;
set reservation_request_timeout = 30000;   -- milliseconds to wait for a slot
```

Common mistake: one shared pool for dashboards and ETL. A single heavy ETL query then fills the pool and queues every interactive user. Separate pools with independent caps keep the SLAs apart.

### Diagnosing failed queries

**Mental model.** Most failures are resource or data problems, not SQL parser problems. Classify the message first: memory limit exceeded (build side too large, wrong join strategy), connection or timeout (one slow node, network), decode error (bad file, wrong delimiter, wrong `FIELDS TERMINATED BY`), or an illegal instruction (codegen assumption about the CPU).

**Worked example.** If the log mentions an illegal instruction, turn off native code generation for one query instead of restarting the daemon.

```sql
set disable_codegen = true;
-- then rerun the failing query; if it passes, the issue is codegen/CPU specific
```

Common mistake: rerunning blindly. Pull the query profile, find the operator with the highest `TotalTime` or the largest `RowsSpilled`, and fix that operator.

## Trade-offs: when to use Impala

**Mental model.** Choose by latency budget, data mutability, and file format. Impala serves sub-second to tens-of-seconds SQL over Parquet, ORC, Kudu, and Iceberg. Hive on Tez or Spark does heavy multi-stage ETL where jobs run for minutes to hours. MapReduce is legacy for new work.

**Gotcha.** Do not run heavy ETL transformations in Impala just because the SQL looks convenient. Every intermediate `INSERT ... SELECT` writes a full copy to HDFS and re-runs statistics collection. Land curated data with Spark or Hive, then serve it from Impala.

**Mental model.** Impala's design point is interactive analytics: ad-hoc SQL, dashboards, and BI extracts over large columnar data. Kudu and Iceberg extend it toward operational analytics with key lookups and row-level change, which is what makes near-real-time dashboards possible.

**Gotcha.** Concurrency is the limit, not single-query speed. Hundreds of simultaneous queries need resource pools and memory caps; without them, latency degrades for everyone. Decide the SLA, then size pools from it.

**Mental model.** Impala trades flexibility for scan speed. It has no multi-statement transactions on HDFS tables, no secondary indexes, a small set of writable formats, and memory-hungry operators. In exchange it gives predictable interactive latency over columnar data with a mature planner and admission control.

**Gotcha.** The trade-off that surprises teams most is concurrency versus query size. Impala is happy with many small queries or a few large ones, but mixing them without pools makes both slow. Decide the split before launch; retrofitting pools after an incident is harder.

When NOT to use Impala: OLTP writes (use PostgreSQL/Kudu-backed services), long multi-stage shuffles (Spark/Hive/Tez), sub-second stateful streaming (Flink), and text search (a search engine).

## Things to remember

### Key numbers and limits

| Topic | Fact | Why it matters |
|---|---|---|
| Architecture | every query runs as an MPP fragment graph across impalad daemons; no MapReduce | low-latency interactive SQL comes from skipping the MR batch machinery — and losing its fault tolerance |
| Catalog daemons | statestored pushes catalog/metadata deltas; catalogd owns the metadata | `INVALIDATE METADATA` (catalog changed) vs `REFRESH` (data files changed) is the daily operator reflex |
| Statistics | the CBO plans from `COMPUTE STATS` output | a table loaded without stats gets the wrong join strategy silently |
| Joins | broadcast (small build side replicated) vs partitioned (both sides shuffled) | the misestimate that matters most is the build side exceeding broadcast range |
| Memory | aggregation/join spilling; admission control queues cap concurrent memory | query failure under load is usually admission control, not bad SQL |
| File layout | Parquet is the default; many small files hurt planning and scan | compact on ingest, partition for pruning |
| Mutability | HDFS/Parquet tables are bulk-optimized (no per-row UPDATE); Kudu is the mutable counterpart | point updates mean you chose Kudu, or you rewrite partitions |

### Semantics to memorize

* **Interactive, not batch.** Impala answers in seconds by holding a query's fragments in memory across daemons — a mid-query node failure kills the query, which is why long ETL stayed on Hive.
* **Metadata is cached aggressively.** Writers creating files outside Impala must be followed by REFRESH/INVALIDATE METADATA, or Impala plans against a stale catalog — the number-one 'my new data doesn't appear' cause.
* **Partition pruning happens against the directory tree.** A query without a predicate on the partition column scans every file.
* **Stats are per-column and per-partition.** Incremental stats keep ANALYZE cheap on growing tables; stale NDV counts flip join order.
* **Spill-to-disk saves big joins from failing — at a price.** Scratch usage in PROFILE is the tell that a query is one size up from its memory envelope.

### Mistakes that keep coming back

* Forgetting `REFRESH` after an external load — the table looks empty or stale in Impala while Hive sees the data fine.
* Skipping `COMPUTE STATS` after bulk loads — broadcast vs partitioned decisions go wrong and giant tables get broadcast.
* Broadcasting an over-large build side — switch strategy or stats first, not more memory.
* Thousands of small files from micro-batch writes — compaction jobs and `INSERT INTO` rewrites are the fix, not more daemons.
* Assuming UPDATE works on a Parquet table — it does not; use Kudu or rewrite the partition.
* Letting one huge query eat the cluster — admission control pools exist to keep interactive SLAs alive under batch load.

## Real world use cases

### 1. Interactive BI over a partitioned Parquet lake

A retailer's analysts query three years of orders because Hive reports take minutes. ~2 PB in HDFS, ~4 billion rows, daily partitions. Spark writes Parquet; Impala serves Hue/Tableau; `catalogd` broadcasts metadata after ETL.

```sql
compute stats analytics.fact_orders;

select order_date, sum(amount) as revenue
from analytics.fact_orders
where order_date between '2026-09-01' and '2026-09-30'
group by order_date order by order_date;
```

Why: pruning keeps a month-scoped query off 97% of the data, and statistics let the optimizer choose a sane plan without hints.

### 2. Operational dimension store on Kudu

A subscription product needs a customer table updated continuously by a service while analysts join it from SQL. ~50 million customers, ~20k upserts/minute. The service writes Kudu; Parquet facts join the Kudu dimension.

```sql
upsert into analytics.dim_customer (customer_id, email, plan, updated_at)
values (1001, 'ada@example.com', 'pro', now());

select customer_id, plan from analytics.dim_customer where customer_id = 1001;
```

Why: primary-key upserts and millisecond lookups, plus SQL access to the same mutable table without a second copy.

### 3. Ad-hoc exploration for a data science team

Data scientists slice clickstream data without waiting for Spark scheduling. ~800 TB, ~120 billion events, hourly partitions on the busiest flows, a dedicated pool, nightly compaction.

```sql
set request_pool = 'ds_ad_hoc';
set mem_limit = 32gb;

select device_type, country, count(*) as events
from analytics.fact_events
where event_date = '2026-10-04' and event_hour between '09' and '17'
group by device_type, country order by events desc limit 50;
```

Why: an isolated pool stops one exploratory query blocking BI, and hour-level pruning makes a huge table interactive.

**How to answer these in an interview.** For each scenario, name the storage format, the partition key, the join strategy, and the guardrail. A strong answer sounds like: Parquet facts partitioned by day, ZSTD compression, 256 MB files, incremental stats after each load, Kudu for the mutable dimension, broadcast joins for the small dimension, and a dedicated BI pool with `MEM_LIMIT` caps.

## Impala exercises

### Try it 1. Broadcast or partitioned join?

**Task.** A 2-million-row dimension is 80 bytes per row. The fact table has 4 billion rows. Broadcast is accepted when the estimated build side is at most 512 MB; otherwise use a partitioned join. Compute the build-side size and choose the strategy.

**Hint.** `build_mb = rows * bytes_per_row / 1024 ** 2`, then compare with the budget.

**Expected result.** Build side `152.6 MB`, so the verdict is `broadcast`.

In [ ]:
BUILD_BUDGET_MB = 512
dim_rows, dim_bytes = 2_000_000, 80
fact_rows = 4_000_000_000

build_mb = dim_rows * dim_bytes / 1024 ** 2
strategy = "broadcast" if build_mb <= BUILD_BUDGET_MB else "partitioned"

print(f"dimension build side = {build_mb:,.1f} MB")
print(f"fact rows = {fact_rows:,} -> strategy: {strategy}")

assert round(build_mb, 1) == 152.6
assert strategy == "broadcast"

### Try it 2. Detect a cardinality misestimate

**Task.** Given estimated and actual row counts for three operators, compute the actual/estimated ratio and flag every operator whose ratio is at least 10. Report the flagged operator and its ratio.

**Hint.** Use `actual / max(estimated, 1)` so a zero estimate does not divide by zero.

**Expected result.** One flag: `HASH JOIN dim_customer` with a ratio of `48.0`.

In [ ]:
operators = [
    ("SCAN HDFS fact_orders", 12_000_000, 12_400_000),
    ("HASH JOIN dim_customer", 50_000, 2_400_000),
    ("AGGREGATE finalize", 8, 8),
]

flagged = []
for name, estimated, actual in operators:
    ratio = actual / max(estimated, 1)
    print(f"{name:<24} estimated={estimated:>10,} actual={actual:>10,} ratio={ratio:6.1f}")
    if ratio >= 10:
        flagged.append((name, round(ratio, 1)))

assert flagged == [("HASH JOIN dim_customer", 48.0)]
print("misestimated operators:", flagged)

### Try it 3. File count for a target size

**Task.** A partition holds 190 GB. Compute how many files it should contain if the target Parquet file size is 256 MB, then compute the average file size. Flag the layout as small-file-bound when the average is below 128 MB.

**Hint.** Use ceiling division: `-(-total_mb // target_mb)`.

**Expected result.** `760` files at `256.0 MB` each, so the layout is healthy.

In [ ]:
partition_gb = 190
TARGET_FILE_MB = 256
SMALL_FILE_MB = 128

total_mb = partition_gb * 1024
files = -(-total_mb // TARGET_FILE_MB)
avg_file_mb = total_mb / files
small_file_bound = avg_file_mb < SMALL_FILE_MB

print(f"{partition_gb} GB / {TARGET_FILE_MB} MB -> {files} files")
print(f"average file size = {avg_file_mb:.1f} MB")
print("small-file-bound:", small_file_bound)

assert files == 760
assert round(avg_file_mb, 1) == 256.0
assert small_file_bound is False

### Try it 4. Size a resource pool

**Task.** Ten workers each give Impala 200 GB, and the pool may use 80 percent of that. A typical dashboard query needs 20 GB and a typical ETL query needs 60 GB. Compute the pool memory and the maximum concurrency for each query shape.

**Hint.** `pool_gb = nodes * mem_per_node_gb * 0.8`; concurrency is integer division by the per-query memory.

**Expected result.** Pool `1600.0 GB`, `80` dashboard queries or `26` ETL queries.

In [ ]:
nodes = 10
mem_per_node_gb = 200
POOL_SHARE = 0.8

pool_gb = nodes * mem_per_node_gb * POOL_SHARE
dashboard_concurrency = int(pool_gb // 20)
etl_concurrency = int(pool_gb // 60)

print(f"pool memory = {pool_gb:,.1f} GB")
print(f"dashboard (20 GB) concurrency = {dashboard_concurrency}")
print(f"ETL (60 GB) concurrency       = {etl_concurrency}")

assert round(pool_gb, 1) == 1600.0
assert dashboard_concurrency == 80
assert etl_concurrency == 26

### Try it 5. Compute spill risk for an operator

**Task.** A hash join build side is 40 GB, the aggregate needs 12 GB, and the sort needs 18 GB. The per-query limit is 32 GB and the scratch limit is 100 GB. Identify which operators must spill, total the estimated scratch space, and decide whether the scratch limit is enough.

**Hint.** An operator spills when its memory exceeds `MEM_LIMIT_GB`; the scratch it needs is the excess over the limit.

**Expected result.** Only `hash join build` spills, needing `8 GB` of scratch, which fits under the `100 GB` limit.

In [ ]:
MEM_LIMIT_GB = 32
SCRATCH_LIMIT_GB = 100
operators = {"hash join build": 40, "aggregate": 12, "sort": 18}

spilling = {}
for name, mem_gb in operators.items():
    if mem_gb > MEM_LIMIT_GB:
        spilling[name] = mem_gb - MEM_LIMIT_GB

scratch_gb = sum(spilling.values())
print("spilling operators:", spilling)
print(f"scratch needed = {scratch_gb} GB, limit = {SCRATCH_LIMIT_GB} GB")

assert list(spilling) == ["hash join build"]
assert scratch_gb == 8
assert scratch_gb <= SCRATCH_LIMIT_GB

### Try it 6. Choose a partition column

**Task.** Query logs show how often each column appears in a filter. A good partition column is filtered often and has a moderate number of distinct values. Choose the column with the most filters among the candidates whose cardinality is at most 10000.

**Hint.** Filter the candidates first, then take the one with the highest filter count.

**Expected result.** Partition by `order_date` (120 filters, 3650 distinct values).

In [ ]:
filter_counts = {"order_date": 120, "customer_id": 90, "region": 40, "status": 25}
cardinality = {"order_date": 3650, "customer_id": 2_000_000, "region": 12, "status": 6}
MAX_CARDINALITY = 10_000

candidates = {c: filter_counts[c] for c in filter_counts if cardinality[c] <= MAX_CARDINALITY}
chosen = max(candidates, key=candidates.get)
print("eligible:", sorted(candidates))
print(f"chosen partition column = {chosen} "
      f"({filter_counts[chosen]} filters, {cardinality[chosen]} distinct values)")

assert chosen == "order_date"
assert cardinality[chosen] <= MAX_CARDINALITY

### Try it 7. Estimate statistics-collection cost

**Task.** A table has 4 billion rows at 120 bytes per row and spans 1095 daily partitions. The cluster collects statistics at 4 million rows per second. Compute the full-collection time in minutes and the per-partition time in seconds.

**Hint.** `seconds = rows / rows_per_second`, then divide by the partition count for the incremental case.

**Expected result.** Full collection takes about `16.7` minutes; one partition takes about `0.9` seconds.

In [ ]:
rows = 4_000_000_000
avg_row_bytes = 120
rows_per_second = 4_000_000
partitions = 1095

full_seconds = rows / rows_per_second
partition_seconds = full_seconds / partitions

print(f"table size = {rows * avg_row_bytes / 1024 ** 3:,.0f} GB")
print(f"full COMPUTE STATS   = {full_seconds / 60:.1f} minutes")
print(f"one partition        = {partition_seconds:.2f} seconds")

assert round(full_seconds / 60, 1) == 16.7
assert round(partition_seconds, 2) == 0.91

### Try it 8. Decide whether a query is shuffle-bound

**Task.** A query scans 1.9 GB but produces an exchange carrying 1.2 billion rows at 40 bytes per row. Compare the exchange volume with the scan volume and decide whether the query is scan-bound or shuffle-bound.

**Hint.** `exchange_gb = rows * bytes_per_row / 1024 ** 3`; compare it with the scan size.

**Expected result.** The exchange carries `44.7 GB` versus a `1.9 GB` scan, so the query is `shuffle-bound`.

In [ ]:
scan_gb = 1.9
exchange_rows = 1_200_000_000
exchange_bytes_per_row = 40

exchange_gb = exchange_rows * exchange_bytes_per_row / 1024 ** 3
verdict = "shuffle-bound" if exchange_gb > scan_gb else "scan-bound"

print(f"scan     = {scan_gb} GB")
print(f"exchange = {exchange_gb:.1f} GB")
print("verdict:", verdict)

assert round(exchange_gb, 1) == 44.7
assert verdict == "shuffle-bound"